In [2]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings('ignore')

In [6]:

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

In [7]:

target_col = [c for c in train.columns if c not in test.columns][0]
id_col = sample_sub.columns[0]
sub_target = sample_sub.columns[1]

is_classification = train[target_col].nunique() < 20 or train[target_col].dtype == 'object'

if is_classification and train[target_col].dtype == 'object':
    le = LabelEncoder()
    train[target_col] = le.fit_transform(train[target_col])

cat_cols = train.drop(columns=[id_col, target_col], errors='ignore').select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    train[col] = train[col].fillna('Missing').astype(str)
    test[col] = test[col].fillna('Missing').astype(str)

X = train.drop(columns=[id_col, target_col], errors='ignore')
y = train[target_col]
X_test = test.drop(columns=[id_col], errors='ignore')

In [ ]:

n_splits = 5
preds = np.zeros(len(X_test))
oof = np.zeros(len(X))

if is_classification:
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    if train[target_col].nunique() > 2:
        preds = np.zeros((len(X_test), train[target_col].nunique()))
else:
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y if is_classification else None)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    
    if is_classification:
        model = CatBoostClassifier(
            iterations=2500, 
            learning_rate=0.03, 
            depth=6, 
            cat_features=cat_cols, 
            random_seed=42, 
            verbose=200
        )
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=100)
        
        if train[target_col].nunique() > 2:
            preds += model.predict_proba(X_test) / n_splits
        else:
            preds += model.predict_proba(X_test)[:, 1] / n_splits
    else:
        model = CatBoostRegressor(
            iterations=2500, 
            learning_rate=0.03, 
            depth=6, 
            cat_features=cat_cols, 
            random_seed=42, 
            verbose=200
        )
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=100)
        preds += model.predict(X_test) / n_splits

0:	learn: 1.0738122	test: 1.0738793	best: 1.0738793 (0)	total: 703ms	remaining: 29m 15s
200:	learn: 0.5689496	test: 0.5731122	best: 0.5731122 (200)	total: 47.8s	remaining: 9m 6s
400:	learn: 0.5580626	test: 0.5662972	best: 0.5662972 (400)	total: 1m 33s	remaining: 8m 7s
600:	learn: 0.5506619	test: 0.5637481	best: 0.5637481 (600)	total: 2m 20s	remaining: 7m 22s
800:	learn: 0.5445841	test: 0.5626386	best: 0.5626386 (800)	total: 3m 7s	remaining: 6m 37s
1000:	learn: 0.5393009	test: 0.5622073	best: 0.5622031 (991)	total: 3m 54s	remaining: 5m 51s
1200:	learn: 0.5343655	test: 0.5619185	best: 0.5618887 (1187)	total: 4m 42s	remaining: 5m 5s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.5618363121
bestIteration = 1289

Shrink model to first 1290 iterations.
0:	learn: 1.0736026	test: 1.0735628	best: 1.0735628 (0)	total: 220ms	remaining: 9m 10s
200:	learn: 0.5689560	test: 0.5733201	best: 0.5733201 (200)	total: 46.8s	remaining: 8m 55s
400:	learn: 0.5580087	test: 0.5666930	best:

In [ ]:

if is_classification and train[target_col].nunique() > 2:
    final_preds = np.argmax(preds, axis=1)
    if 'le' in locals():
        final_preds = le.inverse_transform(final_preds)
    sample_sub[sub_target] = final_preds
else:
    if 'le' in locals() and is_classification:
        sample_sub[sub_target] = le.inverse_transform((preds > 0.5).astype(int))
    else:
        sample_sub[sub_target] = preds

sample_sub.to_csv('submission.csv', index=False)
sample_sub.head()